# ainvoke 요청 응답

FastAPI 라우트 아래에서 호출되는 실제 LangGraph `ainvoke` 결과를 확인합니다.

## 실행 환경 준비

현재 폴더나 상위 폴더에서 `labs/part7/ch30-fastapi-serving`을 찾아 작업 폴더로 사용합니다. 비밀 키를 출력하지 않습니다.

### 이 준비 셀이 하는 일

이 준비 셀은 `Path.cwd().resolve()`로 현재 폴더를 절대 경로로 읽고, 현재 폴더와 부모 폴더에서 `agent_service`가 있는 첫 위치를 `next(..., None)`로 찾습니다. 찾지 못하면 `assert`가 실행을 중단합니다. 찾은 뒤 `os.chdir(lab)`이 이후 import와 파일 접근의 기준 폴더를 바꾸고, 마지막 출력은 실제 선택된 폴더 이름을 확인합니다.

In [1]:
import os
from pathlib import Path
start = Path.cwd().resolve()
lab = next((p for p in (start, *start.parents)
    if (p / 'agent_service').is_dir()), None)
assert lab is not None, 'agent_service가 있는 Ch30 실습 폴더에서 실행하세요.'
os.chdir(lab)
print('실습 폴더:', Path.cwd().name)

실습 폴더: ch30-fastapi-serving


## 기본 실행

컴파일된 그래프에 메시지 상태를 비동기로 전달합니다.

### 바로 다음 코드를 읽는 방법

`await agent.ainvoke(state)`는 상태 하나를 입력해 그래프가 끝날 때까지 기다린 뒤 최종 상태 사전을 반환합니다. `await`는 현재 요청의 결과가 필요하다는 뜻이면서, 기다리는 동안 이벤트 루프가 다른 작업을 처리할 수 있게 하는 지점입니다. 이 예제의 `state`에는 `messages`가 있고, 반환값의 `answer`와 `messages`는 각각 공개할 답과 실행에 사용된 상태를 나타냅니다. 화면에 `print`한 값은 반환 사전 전체가 아니라 학습을 위해 선택한 일부입니다.

In [2]:
from agent_service.agent import agent
state = {'messages': ['운영 시간을 알려 주세요']}
result = await agent.ainvoke(state)
print(result['answer'])
assert '운영 시간을 알려 주세요' in result['answer']

질문을 확인했습니다: 운영 시간을 알려 주세요


## 반환 상태 관찰

최종 상태에는 입력 메시지와 노드가 만든 답이 함께 남습니다.

In [3]:
print(sorted(result))
print(result['messages'])
assert sorted(result) == ['answer', 'messages']

['answer', 'messages']
['운영 시간을 알려 주세요']


## 실패 재현

서비스 규칙이 오류 문구를 만나면 호출 자체가 예외로 끝납니다.

In [4]:
try:
    await agent.ainvoke({'messages': ['오류를 발생']})
except RuntimeError as exc:
    print(type(exc).__name__, str(exc))
    failed = True
assert failed

RuntimeError 교육용 스트림 오류


## 수정

HTTP 스트림 계층에서는 이 예외를 `error` 이벤트로 바꿉니다. 여기서는 정상 질문으로 호출을 마칩니다.

### 이 단계에서 구분할 범위

단일 응답과 스트리밍은 같은 Agent를 소비하는 두 방식입니다. 최종 결과가 한 번에 필요한 저장·평가 작업에는 `ainvoke`가 맞고, 중간 조각을 사용자에게 보여 주려면 다음 절의 `astream`을 사용합니다. 예외가 발생했을 때 내부 예외 문자열을 그대로 HTTP 응답으로 내보내지 않고, 로그의 원인 정보와 공개 오류 코드를 분리합니다.

In [5]:
fixed = await agent.ainvoke({'messages': ['정상 질문']})
print(fixed['answer'])
assert fixed['answer'].endswith('정상 질문')

질문을 확인했습니다: 정상 질문
